# PSOLA-isolated Yorùbá TONE listening test (thin runner)

The **rigorous successor** to nb15's pilot. nb15 judged ✓/✗ on real *TTS* clips, which **confounds tone with
synthetic naturalness** (a ✗ could mean "wrong pitch" OR "robotic voice") and has **no ground truth**. This
kit removes the confound: it manipulates **real native** clips so a `correct` and a `wrong-tone` twin differ
**only in the F0 of one syllable**, both equally PSOLA-processed. **Ground truth is known by construction.**

All logic lives in `pilot/build_psola_form.py` + `pilot/score_psola.py` (which reuse `build_pilot_form.py`'s
S3/HTML/`tone_i2` machinery and `tone_metric`'s PSOLA primitives). This notebook just:
1. installs deps + clones the repo,
2. builds → `pilot/psola_form.html` (self-contained, blind) + `pilot/keymap_psola.json` (withheld GT),
3. **plays 4-6 correct-vs-flipped pairs inline** so you/a native confirm each flip really sounds like **WRONG
   TONE** (not just an artifact) before sending,
4. (optional) scores a reviewer's pasted answers.

**GPU:** none needed, but the MMS-yor aligner is slow on CPU — a **T4/L4** makes the build much faster.

## 1. Install + restart (numpy<2, scoring stack, parselmouth, tone-metric, ffmpeg)

In [ ]:
%pip install --quiet "numpy<2"
%pip install --quiet boto3 soundfile soxr librosa speechbrain "swift-f0" pyworld praat-parselmouth transformers safetensors "huggingface_hub>=0.24.0" torchaudio tqdm
%pip install --quiet --no-deps "git+https://github.com/mosesdaudu001/tone-on-a-budget.git"
%pip uninstall -y hf-xet
import subprocess; subprocess.run(["apt-get","-qq","install","-y","ffmpeg"], check=False)
import os
print("Installs done. RESTARTING so numpy<2 takes effect — run from the NEXT cell.", flush=True)
os._exit(0)

In [ ]:
import numpy as np
assert np.__version__.startswith("1."), "numpy<2 pin did not take — re-run install + restart"
import os, shutil, subprocess
REPO = "/content/tone-on-a-budget" if os.path.isdir("/content") else "/tmp/tone-on-a-budget"
if os.path.isdir(REPO): shutil.rmtree(REPO)
subprocess.run(["git","clone","--depth","1","https://github.com/mosesdaudu001/tone-on-a-budget.git",REPO], check=True)
PILOT = os.path.join(REPO, "pilot")
assert os.path.exists(os.path.join(PILOT,"build_psola_form.py")), PILOT
# sanity: parselmouth must import (the PSOLA engine)
import parselmouth; print("parselmouth", parselmouth.VERSION if hasattr(parselmouth,'VERSION') else 'ok', "| pilot dir:", PILOT)

## 2. Secrets → env (so the build script's `_secret` finds them without prompting)

In [ ]:
import os, getpass
def _secret(k):
    try:
        from google.colab import userdata
        v = userdata.get(k)
        if v: return v
    except Exception: pass
    return os.environ.get(k) or getpass.getpass(f"{k}: ")
os.environ["AWS_ACCESS_KEY_ID"] = _secret("AWS_ACCESS_KEY_ID")
os.environ["AWS_SECRET_ACCESS_KEY"] = _secret("AWS_SECRET_ACCESS_KEY")
os.environ["AWS_DEFAULT_REGION"] = os.environ.get("AWS_DEFAULT_REGION", "us-east-1")
_hf = _secret("HF_TOKEN"); os.environ["HF_TOKEN"] = _hf or ""
if _hf:
    from huggingface_hub import login; login(token=_hf)
os.environ["HF_HUB_DISABLE_XET"] = "1"
print("secrets in env.")

## 3. (optional) DRY RUN — list the native clips it will consider; manipulate / score NOTHING

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, os.path.join(PILOT,"build_psola_form.py"),
                "--dry-run", "--n-clips","14","--n-catch","6"],
               check=True, env=os.environ.copy())

## 4. BUILD — download native, PSOLA-flip one syllable per clip, score tone_i2, embed audio

Writes `pilot/psola_form.html` (blind) + `pilot/keymap_psola.json` (withheld) **inside the cloned repo**.
`--k 1.25` = the flip is 1.25× the clip's own H↔L distance (decisive). Bump `--k` if §5 flips sound too subtle.

In [ ]:
import subprocess, sys
OUT = PILOT
subprocess.run([sys.executable, os.path.join(PILOT,"build_psola_form.py"),
                "--n-clips","14","--k","1.25","--k-catch","2.5","--n-catch","6",
                "--seed","4242","--out-dir",OUT],
               check=True, env=os.environ.copy())
HTML = os.path.join(OUT,"psola_form.html"); KEYMAP = os.path.join(OUT,"keymap_psola.json")
print("\nbuilt:", HTML, "(%.2f MB)" % (os.path.getsize(HTML)/1e6))

## 5. EAR-CHECK (decisive) — play 4-6 correct-vs-flipped pairs inline

For each pair you hear the **correct** twin (plain PSOLA roundtrip) then the **flipped** twin (one syllable's
tone pushed past the opposite band). Both carry the **same** resynthesis artifact, so the only difference is
that one syllable's pitch. **The flipped clip must genuinely sound like WRONG TONE** — if it just sounds like a
glitch / the same word, the flip is too subtle: raise `--k` in §4 and rebuild. (Here, unlike the blind form, the
labels are shown so you can compare.)

In [ ]:
import json, re, base64, tempfile, IPython.display as ipd
html = open(HTML, encoding="utf-8").read()
ITEMS = {it["item_id"]: it for it in json.loads(re.search(r"const ITEMS = (\[.*?\]);", html, re.S).group(1))}
KEY = json.load(open(KEYMAP, encoding="utf-8"))
# group non-catch items by pair_id -> {condition: item_id}
by_pair = {}
for iid, v in KEY.items():
    if not v["is_catch"]:
        by_pair.setdefault(v["pair_id"], {})[v["condition"]] = iid
pairs = [(pid, d["correct"], d["flipped"]) for pid, d in by_pair.items() if "correct" in d and "flipped" in d][:6]
tmp = tempfile.mkdtemp()
def _play(iid, tag):
    it = ITEMS[iid]; header, b64 = it["audio"].split(",", 1)
    ext = "mp3" if "mpeg" in header else "wav"
    fp = f"{tmp}/{iid}.{ext}"; open(fp,"wb").write(base64.b64decode(b64))
    ipd.display(ipd.HTML(f"<b>{tag}</b> &nbsp; {ITEMS[iid]['text']}"))
    ipd.display(ipd.Audio(fp))
for pid, cid, fid in pairs:
    km = KEY[fid]
    ipd.display(ipd.HTML(f"<hr><h4>{pid} — TBU #{km['tbu_index']}, flip {km['flip_dir']} "
                         f"({km['semitones']:+.2f} st)</h4>"))
    _play(cid, "✓ CORRECT (roundtrip)")
    _play(fid, "✗ FLIPPED (wrong tone)")
print(f"\nPlayed {len(pairs)} pairs. If each FLIPPED clip sounds like WRONG TONE, the kit is ready to send.")

## 6. Download the form to send (Colab)

Send `psola_form.html` to a native reviewer. `keymap_psola.json` is the **withheld key — never send it.**

In [ ]:
try:
    from google.colab import files; files.download(HTML)
except Exception:
    print("(not on Colab) the form is at:", HTML)

## 7. (optional) SCORE a reviewer's pasted answers (known ground truth)

Paste the reviewer's `PILOT1;item01=ok;...` code into `ANSWERS`. Run with no answers to get the **human-free
metric sensitivity** (paired oracle) alone.

In [ ]:
import subprocess, sys
ANSWERS = ""   # <- paste the reviewer's pasted code here, e.g. "PILOT1;item01=ok;item02=bad;..."
cmd = [sys.executable, os.path.join(PILOT,"score_psola.py"), "--keymap", KEYMAP]
if ANSWERS.strip():
    cmd += ["--answers", ANSWERS]
subprocess.run(cmd, check=True, env=os.environ.copy())